# Project 1: Medallion Pipeline
This notebook builds the medallion pipeline one step at a time.

- **Bronze** - raw data, exactly as received. No cleaning here.
- **Silver** - cleaned and deduplicated, using MERGE.
- **Gold** - business-ready aggregates.

<DETAILS>
<SUMMARY>IN THIS NOTEBOOK</SUMMARY>

* **Setup & Config**: Spark Session tuned for Delta Lake.  
* **Bronze Layer**: Automated download & raw Delta landing.  
* **Silver Layer**: Data cleansing, deduplication, surrogate key generation (trip_id), and an idempotent Delta MERGE.  
* **Gold Layer**: Business aggregation for reporting/dashboards.    
* **Audit / Quality Gate**: Automated data and financial reconciliation.  
* Data -> [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
</DETAILS>

## Step 0 - setup
- create a spark session with delta extaension and define ouptut paths

In [1]:
import os
%pip install -q pyspark delta-spark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

spark = (configure_spark_with_delta_pip(
    SparkSession.builder.master("local[2]")
    .appName("Project_1")
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions","4")
    .config("spark.ui.showConsoleProgress","false")
)).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

BRONZE = os.path.expanduser("~/projects/data-engineering/Project_1/proj1_bronze")
SILVER = os.path.expanduser("~/projects/data-engineering/Project_1/proj1_silver")
GOLD = os.path.expanduser("~/projects/data-engineering/Project_1/proj1_gold")

Note: you may need to restart the kernel to use updated packages.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 23:22:38 WARN Utils: Your hostname, Kirika, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/30 23:22:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/josep/projects/data-engineering/Project_1/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/josep/.ivy2.5.2/cache
The jars for the packages stored in: /home/josep/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d45fa557-8dc1-42a5-ac36-e36e770d8e61;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in c

- Spin up a Spark engine configured to work with Delta Lake tables.
- Read raw files using an enforced schema (so errors fail fast).
- Transform the columns using Spark functions (F).

## Step 1: Bronze (raw data as received)
write the raw orders table into bronze Delta table

In [2]:
import urllib.request

#1. define the url and bronze landing path
year, month = "2025","01"
url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet"

raw_landing_folder = os.path.join(BRONZE,"raw_landing")
os.makedirs(raw_landing_folder,exist_ok=True)

local_parquet_path = os.path.join(raw_landing_folder,f"yellow_{year}_{month}.parquet")
delta_bronze_table = os.path.join(BRONZE,"yellow_taxi_delta")

#2. download,  only if  not on disk, the parquet file from the url and save it to the bronze landing path
if not os.path.exists(local_parquet_path):
    print(f"Downloading {url} to {local_parquet_path}...")
    urllib.request.urlretrieve(url, local_parquet_path)
else:
    print(f"File {local_parquet_path} already exists. Skipping download.")

#3. read the parquet file into a Spark DataFrame
src = os.path.basename(local_parquet_path) #eg. yellow_2025_01.parquet

df = (
    spark.read.parquet(local_parquet_path)
    .withColumn("_source_file",F.lit(src))
    .withColumn("_ingest_at",F.current_timestamp())
    )

#4. INDEMPOTENT WRITE: replace only rows for this source file
(
    df.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere",f"_source_file = '{src}'")
    .option("overwriteSchema","true")
    .save(delta_bronze_table)
)

print(f"Ingested {df.count():,} rows into the Bronze Delta for {src}")

File /home/josep/projects/data-engineering/Project_1/proj1_bronze/raw_landing/yellow_2025_01.parquet already exists. Skipping download.
Ingested 3,475,226 rows into the Bronze Delta for yellow_2025_01.parquet


**pipeline code fetches the file dynamically**, lands it in your BRONZE zone as raw data, and Spark immediately picks it up:

<details>
<summary>Method 2: The Zero-Disk Way (Stream over HTTP directly into Memory)</summary>
  
If you have strict disk space limits and want to read the URL directly over the internet without saving any intermediate `.parquet` file to your drive, you can use **DuckDB** with **PyArrow** as a bridge.


DuckDB can read remote HTTP Parquet files on the fly using range requests:
```
pip install duckdb pyarrow
import duckdb

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

# 1. Read remote HTTPS directly into an in-memory Apache Arrow table
arrow_table = duckdb.query(f"SELECT * FROM '{url}'").arrow()

# 2. Turn the Arrow table into a Spark DataFrame
df = spark.createDataFrame(arrow_table.to_pandas())

# 3. Write directly to your Bronze Delta Lake table
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
df.write.format("delta").mode("append").save(delta_bronze_table)
```
**How it works:** DuckDB pulls only the needed chunks from CloudFront over HTTPS into RAM, converts them to Arrow, hands them to Spark, and Spark writes them directly into your local Delta Lake directory—bypassing the manual download step entirely.
</details>

In [3]:
try:
    import pandas as pd
except ImportError:
    %pip install pandas

import pandas as pd

# Allow full text to display in Colab without truncating
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

data_dict = [
    {"Field Name": "VendorID", "Description": "A code indicating the TPEP provider that provided the record:\n1 = Creative Mobile Technologies, LLC\n2 = Curb Mobility, LLC (VeriFone)\n6 = Myle Technologies Inc.\n7 = Helix"},
    {"Field Name": "tpep_pickup_datetime", "Description": "The date and time when the meter was engaged."},
    {"Field Name": "tpep_dropoff_datetime", "Description": "The date and time when the meter was disengaged."},
    {"Field Name": "passenger_count", "Description": "The number of passengers in the vehicle (driver-entered value)."},
    {"Field Name": "trip_distance", "Description": "The elapsed trip distance in miles reported by the taximeter."},
    {"Field Name": "RatecodeID", "Description": "The final rate code in effect at the end of the trip:\n1 = Standard rate\n2 = JFK\n3 = Newark\n4 = Nassau or Westchester\n5 = Negotiated fare\n6 = Group ride\n99 = Null/unknown"},
    {"Field Name": "store_and_fwd_flag", "Description": "Whether the trip record was stored in vehicle memory before sending because of no server connection:\nY = store and forward trip\nN = not a store and forward trip"},
    {"Field Name": "PULocationID", "Description": "TLC Taxi Zone in which the taximeter was engaged."},
    {"Field Name": "DOLocationID", "Description": "TLC Taxi Zone in which the taximeter was disengaged."},
    {"Field Name": "payment_type", "Description": "A numeric code signifying how the passenger paid:\n0 = Flex Fare trip\n1 = Credit card\n2 = Cash\n3 = No charge\n4 = Dispute\n5 = Unknown\n6 = Voided trip"},
    {"Field Name": "fare_amount", "Description": "The time-and-distance fare calculated by the meter."},
    {"Field Name": "extra", "Description": "Miscellaneous extras and surcharges (e.g., $0.50 rush hour and $1 overnight charges)."},
    {"Field Name": "mta_tax", "Description": "$0.50 MTA tax that is automatically triggered based on the metered rate in use."},
    {"Field Name": "tip_amount", "Description": "Tip amount. Automatically populated for credit card tips. Cash tips are not included."},
    {"Field Name": "tolls_amount", "Description": "Total amount of all tolls paid in trip."},
    {"Field Name": "improvement_surcharge", "Description": "$0.30 improvement surcharge assessed on hailed trips."},
    {"Field Name": "total_amount", "Description": "The total amount charged to passengers. Does not include cash tips."},
    {"Field Name": "congestion_surcharge", "Description": "Total amount collected in trip for NYS congestion surcharge."},
    {"Field Name": "Airport_fee", "Description": "$1.25 fee for pick-up only at LaGuardia and JFK Airports."}
]

df_dict = pd.DataFrame(data_dict)
df_dict

,Field Name,Description
0,VendorID,"A code indicating the TPEP provider that provided the record:\n1 = Creative Mobile Technologies, LLC\n2 = Curb Mobility, LLC (VeriFone)\n6 = Myle Technologies Inc.\n7 = Helix"
1,tpep_pickup_datetime,The date and time when the meter was engaged.
2,tpep_dropoff_datetime,The date and time when the meter was disengaged.
3,passenger_count,The number of passengers in the vehicle (driver-entered value).
4,trip_distance,The elapsed trip distance in miles reported by the taximeter.
5,RatecodeID,The final rate code in effect at the end of the trip:\n1 = Standard rate\n2 = JFK\n3 = Newark\n4 = Nassau or Westchester\n5 = Negotiated fare\n6 = Group ride\n99 = Null/unknown
6,store_and_fwd_flag,Whether the trip record was stored in vehicle memory before sending because of no server connection:\nY = store and forward trip\nN = not a store and forward trip
7,PULocationID,TLC Taxi Zone in which the taximeter was engaged.
8,DOLocationID,TLC Taxi Zone in which the taximeter was disengaged.
9,payment_type,A numeric code signifying how the passenger paid:\n0 = Flex Fare trip\n1 = Credit card\n2 = Cash\n3 = No charge\n4 = Dispute\n5 = Unknown\n6 = Voided trip


## Step 2: Silver (Clean and deduplicate with merge)
- reads bronze, cleans it(dedup by order_id), and merges the cleaned rows into silver.  
- reruns must be safe. MERGE is indempotent, a blind append would double the data.

In [4]:

from pyspark.sql.window import Window as window



#1. Define paths matching Bronze ingestion step
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_quarantine_table = os.path.join(SILVER,"yellow_taxi_quarantine_delta")

#2. Read BRONZE Delta table
bronze_df = spark.read.format("delta").load(delta_bronze_table)

#3. clean  and Enrich for Silver
#   a. create a unique "trip_id" by hashing identifying columns
#   b. Tag invalid records with an explicit failure reason instead of discarding them, and write them to a quarantine table for later inspection

#define a windowpartitioned by trip_id
window_spec = window.partitionBy("trip_id").orderBy("tpep_pickup_datetime")

tagged_df = (
  bronze_df.withColumn("trip_id",
    F.sha2(F.concat_ws("||",
      F.coalesce(F.col("VendorID").cast("string"),F.lit("")),
      F.coalesce(F.col("tpep_pickup_datetime").cast("string"),F.lit("")),
      F.coalesce(F.col("tpep_dropoff_datetime").cast("string"),F.lit("")),
      F.coalesce(F.col("PULocationID").cast("string"),F.lit("")),F.coalesce(F.col("VendorID").cast("string"),F.lit("")),
      F.coalesce(F.col("DOLocationID").cast("string"),F.lit("")),F.coalesce(F.col("VendorID").cast("string"),F.lit("")),
      F.coalesce(F.col("fare_amount").cast("string"),F.lit(""))
    ),256)
  )
  #Assign row number within duplicates(1 = primary copy, >1 = duplicate copy)
  .withColumn("duplicate_rank",F.row_number().over(window_spec))
  .withColumn("quarantine_reason",
               F.when(F.col("duplicate_rank") > 1, "Duplicate record") # <----- catches the duplciates
               .when(F.col("tpep_pickup_datetime").isNull(), "Missing pickup datetime")
               .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"), "Invalid trip duration")
               .when(F.col("trip_distance") < 0, "Negative trip distance")
               .when(F.col("fare_amount") < 0, "Negative fare amount")
               .when(
                   (F.year("tpep_pickup_datetime") != int(year)) |
                    (F.month("tpep_pickup_datetime") != int(month)),
                    "Pickup datetime not in specified year/month")
               .otherwise(None)
               ).drop("duplicate_rank"))


#4. split stream into valid(silver) vs. invalid(quarantine) records
valid_df = (
    tagged_df.filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason")
    .cache() #keeps records in memory
)

quarantine_df = (
  tagged_df.filter(F.col("quarantine_reason").isNotNull())
)


#5. check if Silver Table exists and has data
is_first_load = not DeltaTable.isDeltaTable(spark,delta_silver_table) or(
    spark.read.format("delta").load(delta_silver_table).limit(1).count()==0)

if is_first_load:
  # FASTPATH: direct append on first load which bypassess expensive full join
  print("Silver table empty. Performing fast initial write...")
  valid_df.write.format("delta").mode("overwrite").save(delta_silver_table)
else:
  #Incremental path: merge updates/new rows
  print("Existing data detected; Performing Delta MERFE...")
  target_silver = DeltaTable.forPath(spark,delta_silver_table)
  target_silver.alias("target")\
    .merge(valid_df.alias("source"), "target.trip_id = source.trip_id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

#6. Append Quarantined records directly
print("Writing quarantined records...")
quarantine_df.write.format("delta").mode("append").save(delta_quarantine_table)

#7. Unpersist cache to free RAM
valid_df.unpersist()

#8.  report
silver_count = spark.read.format("delta").load(delta_silver_table).count()
quarantine_count = spark.read.format("delta").load(delta_quarantine_table).count()

print(
   f"{'-'*5} INGESTION REPORT {'-'*5}\n"
   f"Rows in Silver (CLEAN): {silver_count:,}\n"
   f"Rows in Quarantine: {quarantine_count:,}"
)

Existing data detected; Performing Delta MERFE...
Writing quarantined records...
----- INGESTION REPORT -----
Rows in Silver (CLEAN): 3,330,662
Rows in Quarantine: 144,564


* **No Single `order_id`**: NYC TLC records do not come with an order ID. To merge and deduplicate, data engineers create a surrogate trip_id by hashing key columns (VendorID, pickup time, dropoff time, and location IDs).

* **Quality Cleaning (The Silver Standard)**: Real Silver tables filter out bad data—like trips where the dropoff is before the pickup, or trips with negative fares.

* **Don't Overwrite Silver Every Run**: Your original template did .mode("overwrite") with an empty table before the MERGE. If you do that, every new month will wipe out the previous months. We check if the table exists first.
----

1. `F.sha2(F.concat_ws(...), 256)`:
Takes the 6 columns that define a specific ride and hashes them into a single string (e.g.`, "a3f18b..."`). This acts exactly like your order_id.
2. `cleaned.limit(0).write...`:
Instead of writing a manual 20-line StructType schema with `StructField, limit(0)` grabs an empty DataFrame with the exact types inherited from Bronze and creates the table shell in one clean line.
3. `DeltaTable.isDeltaTable(...)`:
Prevents wiping out your Silver table on future runs when you process month `02`, `03`, etc.

##### observations

`import shutil
shutil.rmtree(delta_silver_table, ignore_errors=True)`

**Why**: running locally on your machine or Colab disk (`/root/proj1_silver`). Local Python file operations work normally here.When you deploy this to the cloud (AWS S3 or Azure ADLS via s3://... or abfss://...), shutil.rmtree will fail because Python cannot treat cloud object stores like local hard drive folders

 - In cloud environments, you let Delta overwrite schemas natively:
```p
cleaned.limit(0).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_silver_table)
```

---
##### performance
When running `.merge()`, Spark will evaluate the cleaned dataframe twice **(once to find matching keys**, and **once to read the payload to insert/update**).   
If you notice your Colab cell taking a long time, cache the cleaned records in memory before the merge:
```p
# Cache before the merge to prevent Spark from recomputing hashes and filters
cleaned = cleaned.cache()

# Execute merge
target_table.alias("target") \
  ...
  .execute()

# Unpersist after merge is complete
cleaned.unpersist()
```
#####  adding to Silver
Silver is also where you enrich data with helpful derived columns. Adding these saves time when building the Gold layer:  
* `trip_duration_minutes`
```p   
(F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60
```
* `pickup_date`
```P
F.to_date(F.col("tpep_pickup_datetime"))
```
This date column can also be used as a partition column to make your Gold layer aggregations much faster

## Step 3: Gold (aggregates)
builds the gold aggregates, the business ready layer

In [5]:
#1. define paths
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_gold_table = os.path.join(GOLD,"daily_taxi_summary")

#2. Read the silver delta table
silver_df = spark.read.format("delta").load(delta_silver_table)

#3. Aggregate into Business metrics(the gold transform)
gold_df = (
    silver_df.withColumn("pickup_date",F.to_date("tpep_pickup_datetime"))       #derive pickup date from timestamp
    .groupBy("pickup_date","VendorID")                                      #group by date and vendor
    .agg(                                                                   #calculate executive KPIs
        F.count("*").alias("total_trips"),
        F.sum("passenger_count").alias("total_passengers"),
        F.round(F.sum("total_amount"),2).alias("total_revenue"),
        F.round(F.sum("fare_amount"),2).alias("total_fare"),
        F.round(F.sum("tip_amount"),2).alias("total_tips"),
        F.round(F.avg("trip_distance"),2).alias("avg_distance_miles"),
        F.round(F.avg("fare_amount"),2).alias("avg_fare_per_trip")

)
 .orderBy("pickup_date","VendorID")
)

#4. Save to Gold
#----NB can save as:
#                     "delta" -> recommended  for data lakes
# or
#                     "Parquet" -> for external tools
gold_df.write.format("delta").mode("overwrite").save(delta_gold_table)
print(f"Gold layer written successfully")

#5. preview the dashboard_ready table

print("\n ---GOLD SUMMARY PREVIEW---")
spark.read.format("delta").load(delta_gold_table).show(10,truncate=False)

Gold layer written successfully

 ---GOLD SUMMARY PREVIEW---
+-----------+--------+-----------+----------------+-------------+----------+----------+------------------+-----------------+
|pickup_date|VendorID|total_trips|total_passengers|total_revenue|total_fare|total_tips|avg_distance_miles|avg_fare_per_trip|
+-----------+--------+-----------+----------------+-------------+----------+----------+------------------+-----------------+
|2025-01-01 |1       |17363      |19606           |482067.31    |348605.74 |48918.89  |3.55              |20.08            |
|2025-01-01 |2       |65040      |91367           |1892496.18   |1339599.11|213475.58 |5.07              |20.6             |
|2025-01-01 |6       |13         |NULL            |624.11       |44.95     |0.0       |15.79             |3.46             |
|2025-01-01 |7       |15         |23              |320.15       |185.0     |55.15     |1.71              |12.33            |
|2025-01-02 |1       |18692      |21440           |515580.37    

 The most common business need is a **Daily Performance** & **Revenue Summary** :
 - tracking total revenue
 - trip volume
 - tips
 - average distance per day.    

##### observations
1. Specific Table Path (daily_taxi_summary):   
- In production, you never write directly to the root GOLD folder because your project will eventually have multiple Gold tables (e.g., ***daily_summary***, ***airport_pickups***, ***driver_tip_analysis***). Always create a named subfolder.
2. Metric Rounding (F.round(..., 2)):   
- Financial and business metrics (***total_revenue***, ***avg_fare***) should always be rounded to 2 decimal places to avoid messy floating-point outputs like `$14.20000000004` on dashboards.
3. Delta vs. Parquet for Gold:   
    - **When to use Delta**: If internal teams query Gold using Spark, Trino, or Databricks, use Delta to retain ACID transactions and time travel.
    - **When to use Parquet**: If you need to hand these files directly to older tools or lightweight Python libraries that do not read Delta tables natively.


## Step 4: Check the numbers (Data reconciliation & pipeline auditing )
compare row counts and totals across bronze -> silver -> gold

In [6]:
#1. Define paths matching the ebtire pipeline
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_quarantine_table = os.path.join(SILVER,"yellow_taxi_quarantine_delta")
delta_gold_table = os.path.join(GOLD,"daily_taxi_summary")

#2. extract bronze metrics
bronze_df = spark.read.format("delta").load(delta_bronze_table)
bronze_count = bronze_df.count()
bronze_total = bronze_df.agg(F.sum("total_amount")).collect()[0][0] or 0.0

#3. Extract silver metrics (Row count + Total revenue)
silver_df = spark.read.format("delta").load(delta_silver_table)
silver_count = silver_df.count()
silver_total = silver_df.agg(F.sum("total_amount")).collect()[0][0] or 0.0

#3.1 Extract Quarantine metrics(Dirty/Rejected volume & revenue)
if DeltaTable.isDeltaTable(spark, delta_quarantine_table):
    quarantine_df = spark.read.format("delta").load(delta_quarantine_table)
    quarantine_count = quarantine_df.count()
    quarantine_total = quarantine_df.agg(F.sum("total_amount")).collect()[0][0] or 0.0
else:
    quarantine_count = 0
    quarantine_total = 0.0

#4. Extract Gold metrics (aggregated rows + Toatal revenue)
gold_df = spark.read.format("delta").load(delta_gold_table)
gold_count = gold_df.count()
gold_total = gold_df.agg(F.sum("total_revenue")).collect()[0][0] or 0.0

#5. Calculations and Reconciliations
accounted_rows = silver_count + quarantine_count


accounted_rev = silver_total + quarantine_total
var_bronze_vs_accounted = abs(bronze_total-accounted_rev)
var_silver_vs_gold = abs(silver_total-gold_total)

#6. Print executive Audit report
print(f"{'='*55}\n"
        f"{'Medallion PIPELINE AUTIT REPORT':^55} \n"
        f"{'='*55}\n"
        f"{'-'*15} RAW COUNT INTEGRITY {'-'*15}\n"
        f"Bronze raw inflow rows:                 {bronze_count:,}\n"
        f"Silver Cleaned rows:                    {silver_count:,}\n"
        f"Quarantined (bad) Rows:                 {quarantine_count:,}\n"
        f"Gold aggregate rows:                    {gold_count:,}\n"
        f"\n{'-'*15} FINANCIAL RECONCILIATION {'-'*15}\n"
        f"Bronze raw inflaw revenue:              ${bronze_total:,.2f}\n"
        f"Silver cleaned revenue:                 ${silver_total:,.2f}\n"
        f"Quarantined bad revenue:                ${quarantine_total:,.2f}\n"
        f"Total ACCOUNTED revenue(silver+quarn):  ${accounted_rev:,.2f}\n"
        f"Gold kpi Summary revenue:               ${gold_total:,.2f}\n"
        f"{'-'*55}\n"
        f"Variance (Bronze vs Accounted):         ${var_bronze_vs_accounted:,.2f}\n"
        f"Variance (Silver vs Gold):              ${var_silver_vs_gold:,.2f}\n"
        f"{'='*55}"
        )

#7. Automated Quality Gates (data integrity Assertions)

assert bronze_count > 0, "Audit failed: Bronze layer is empty"
assert silver_count > 0, "Audit failed: Silver layer is empty"
assert silver_count <= bronze_count,  "Audit failed: Silver has more rows than Bronze"
assert gold_count < silver_count, "Audit failed: Gold did not aggreagte data properly"

#8. Financial reconciliation assertions:
#Gate A: allow upto a $1.00 variance due to floating point rounding across the rows(Gold MUST match silver)
assert var_silver_vs_gold < 1.0, (
    f"Audit failed: Financial mismatch! Silver=${silver_total:,.2f} vs Gold=${gold_total:,.2f}"
    )
#Gate B: 100% of Bronze revenue is accouunted for in either Silver or Quarantine (ignoring duplicate records)
assert var_bronze_vs_accounted < 5.0, (
    f"Audit failed: unaccounted revenue! Bronze=${bronze_total:,.2f} vs Accounted=${accounted_rev:,.2f}"
)
print("\n ALL CHECKS PASSED: PIPELINE IS RECONCILED AND HEALTHY")

            Medallion PIPELINE AUTIT REPORT             
--------------- RAW COUNT INTEGRITY ---------------
Bronze raw inflow rows:                 3,475,226
Silver Cleaned rows:                    3,330,662
Quarantined (bad) Rows:                 144,564
Gold aggregate rows:                    124

--------------- FINANCIAL RECONCILIATION ---------------
Bronze raw inflaw revenue:              $89,005,026.80
Silver cleaned revenue:                 $90,330,253.66
Quarantined bad revenue:                $-1,325,226.86
Total ACCOUNTED revenue(silver+quarn):  $89,005,026.80
Gold kpi Summary revenue:               $90,330,253.66
-------------------------------------------------------
Variance (Bronze vs Accounted):         $0.00
Variance (Silver vs Gold):              $0.00

 ALL CHECKS PASSED: PIPELINE IS RECONCILED AND HEALTHY


In [7]:
# Check total amount of all negative/invalid fares in Bronze
bad_bronze_money = bronze_df.filter(F.col("fare_amount") < 0).agg(F.sum("total_amount")).collect()[0][0]
print(f"Total Negative Fares in Bronze: ${bad_bronze_money:,.2f}")

Total Negative Fares in Bronze: $-1,330,081.05


Data has millions of rows that change month-to-month, you **cannot** hardcode static numbers like `assert bronze_count == 5` . Instead, you write ***relational assertions***:

1. `Silver rows <= Bronze rows`: Because deduplication and quality filters remove dirty rows.
2. `Gold Revenue == Silver Revenue`: The total money in the executive dashboard (Gold) ***MUST balance*** with the money in the cleaned records (Silver).

#### This gives you:
 - **Financial Integrity**: prove that aggregation logic didn't accidentally discard money or double-count payments between Silver and Gold.
- **Data Drop Visibility**: It prints the exact percentage of corrupted/duplicate rides cleaned out between Bronze and Silver.
- **Automated Pipeline Gate**: If this runs inside an orchestrator (like Apache Airflow or Databricks Workflows), any AssertionError will immediately fail the pipeline and ***alert on Slack/PagerDuty*** before dirty data reaches dashboards.



## Step 5 - Serve: the Gold layer goes to the consumer

**What this cell does:** reads the gold table (`daily_taxi_summary`) with a serving engine - DuckDB over the Delta table - and previews + totals the revenue.

**Why this matters:** the pipeline is done. Serving is the hand-off from producer (Spark) to consumer (analyst, dashboard, downstream service). Gold is the layer the consumer queries.

**Note:** in the paid world this is where you would point a warehouse (Snowflake) at the gold layer. Free version: serve straight from the Delta table with DuckDB - no warehouse needed.


In [8]:
try:
  import duckdb
except ImportError:
  %pip install -q duckdb

gold_cron = duckdb.connect()

preview = gold_cron.execute(
    f"SELECT * FROM delta_scan('{delta_gold_table}') limit 10;"
).fetchdf()
print(preview)

served_total = gold_cron.execute(
    f"SELECT SUM(total_revenue) FROM delta_scan('{delta_gold_table}')"
).fetchdf().squeeze()
print(f"Served gold revenue: ${served_total:,.2f}")

  pickup_date  VendorID  total_trips  total_passengers  total_revenue  \
0  2025-01-01         1        17363             19606      482067.31   
1  2025-01-01         2        65040             91367     1892496.18   
2  2025-01-01         6           13              <NA>         624.11   
3  2025-01-01         7           15                23         320.15   
4  2025-01-02         1        18692             21440      515580.37   
5  2025-01-02         2        63346             92383     1907227.57   
6  2025-01-02         6           13              <NA>         377.60   
7  2025-01-02         7           18                24         369.52   
8  2025-01-03         1        20209             23541      541235.14   
9  2025-01-03         2        68366             98805     1952388.64   

   total_fare  total_tips  avg_distance_miles  avg_fare_per_trip  
0   348605.74    48918.89                3.55              20.08  
1  1339599.11   213475.58                5.07              20.